# 02 — I segnali per testata, e se c'è segnale (tentativo 09)

Dai segnali per articolo (`01_estrai_segnali`) a una riga per testata, **senza nessuna etichetta**:

- **pagina**: la media sugli articoli della testata (per i segnali sì/no: la quota di articoli che li hanno);
- **link, per tipo**: link citati per articolo, e la quota dei link citati verso social, video (YouTube/Vimeo), video
  alternativi (Rumble, BitChute, Odysee, Brighteon), siti istituzionali (.gov, .edu, .int, .mil, università),
  Wikipedia, e altre testate presenti in NewsGuard (solo la presenza del dominio nell'elenco, non il punteggio).
  I link verso siti dello stesso gruppo si tolgono (sono parentele, non citazioni).

L'elenco completo dei domini citati per testata (per il modello sui link) resta per articolo in
`../Dataset/segnali/link_citati.csv.gz`; l'affidabilità dei siti citati, che usa etichette, si calcola solo in
classificazione e solo con il training di ogni fold.

**C'è segnale?** Solo descrittivo, prima di ogni modello: per ogni segnale la mediana per livello e la correlazione di
Spearman con il punteggio NewsGuard, sia su tutte le 863 testate sia con **un valore per gruppo** (la media del
gruppo), perché un gruppo grande (Natural News) non conti per centinaia. Poi i domini più citati per livello.
Uscite: `../Dataset/segnali/testate.csv.gz`, `../Results/2-segnali/02_*.csv`.

In [ ]:
import re

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from comune import LIVELLI, NEWSGUARD, RESULTS, SEGNALI, VERSIONI

In [ ]:
DOMINI = {
    "social": r"(^|\.)(twitter|x|facebook|instagram|tiktok|linkedin|reddit|truthsocial|gab|gettr|parler|threads)\.(com|net)$|^t\.(me|co)$",
    "video": r"(^|\.)(youtube\.com|youtu\.be|vimeo\.com)$",
    "video alternativi": r"(^|\.)(rumble\.com|bitchute\.com|odysee\.com|brighteon\.com)$",
    "istituzionali": r"(^|\.)(gov|mil|int|edu)(\.[a-z]{2})?$|\.ac\.[a-z]{2}$|(^|\.)(europa\.eu|un\.org|who\.int)$",
    "wikipedia": r"(^|\.)wikipedia\.org$",
}
t = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain")
ng = pd.read_csv(NEWSGUARD, low_memory=False); ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
t["score"] = t.index.map(ng.Score)
pag = pd.read_csv(SEGNALI / "pagine.csv.gz")
lnk = pd.read_csv(SEGNALI / "link_citati.csv.gz")
art = pd.read_csv(VERSIONI / "dataset_finale.csv", usecols=["domain"]); n_art = art.domain.value_counts()
print(f"{len(pag)} articoli, {pag.domain.nunique()} testate; {len(lnk)} coppie articolo-dominio citato")

## Una riga per testata

In [ ]:
gruppo = t.gruppo
# link: via quelli verso siti dello stesso gruppo (il dominio citato è una testata del dataset nello stesso gruppo)
lnk["stesso gruppo"] = lnk.citato.map(gruppo).eq(lnk.domain.map(gruppo))
print(f"link verso siti dello stesso gruppo, tolti: {lnk['stesso gruppo'].mean():.1%}")
lnk = lnk[~lnk["stesso gruppo"]]
noti = set(ng.index)
for k, rx in DOMINI.items():
    lnk[k] = lnk.citato.str.contains(rx, regex=True)
lnk["testate NewsGuard"] = lnk.citato.isin(noti)
pesi = lnk.volte
L = pd.DataFrame({"link citati per articolo": lnk.groupby("domain").volte.sum() / n_art})
for k in list(DOMINI) + ["testate NewsGuard"]:
    L[f"link: quota {k}"] = (lnk[k] * pesi).groupby(lnk.domain).sum() / pesi.groupby(lnk.domain).sum()
P = pag[pag.html == 1].drop(columns=["article_id", "html"]).groupby("domain").mean()
S = P.join(L, how="outer").reindex(t.index)
S["link citati per articolo"] = S["link citati per articolo"].fillna(0)
S.to_csv(SEGNALI / "testate.csv.gz")
print(f"{S.shape[0]} testate x {S.shape[1]} segnali; testate senza nessun link citato: {S['link: quota social'].isna().sum()}")

## C'è segnale? Mediana per livello e correlazione con il punteggio

In [ ]:
liv = pd.cut(t.score, [-1, 39, 59, 74, 99, 100], labels=LIVELLI)
righe = {}
for c in S.columns:
    x = S[c]
    ok = x.notna() & t.score.notna()
    rho = spearmanr(x[ok], t.score[ok]).statistic
    g = pd.DataFrame({"x": x, "s": t.score, "g": gruppo})[ok].groupby("g").mean()
    rho_g = spearmanr(g.x, g.s).statistic
    righe[c] = {**{l: x[liv == l].median() for l in LIVELLI}, "rho (testate)": rho, "rho (un valore per gruppo)": rho_g}
D = pd.DataFrame(righe).T
D = D.reindex(D["rho (un valore per gruppo)"].abs().sort_values(ascending=False).index)
D.round(3).to_csv(RESULTS / "02_segnali_per_livello.csv")
print(f"gruppi: {gruppo.nunique()}; segnali con |rho per gruppo| >= 0,3: {(D['rho (un valore per gruppo)'].abs() >= 0.3).sum()} su {len(D)}")
D.round(2)

## I domini più citati per livello

Per ogni livello: i domini citati dal maggior numero di testate di quel livello (quota di testate), senza contare i
siti dello stesso gruppo.

In [ ]:
cit = lnk.groupby(["domain", "citato"]).volte.sum().reset_index()
cit["livello"] = cit.domain.map(liv)
quante = liv.value_counts()
righe = []
for l in LIVELLI:
    q = cit[cit.livello == l].groupby("citato").domain.nunique().sort_values(ascending=False).head(20) / quante[l]
    righe += [{"livello": l, "dominio citato": d, "quota di testate": round(v, 2)} for d, v in q.items()]
top = pd.DataFrame(righe); top.to_csv(RESULTS / "02_domini_piu_citati_per_livello.csv", index=False)
for l in LIVELLI:
    x = top[top.livello == l]
    print(f"{l}: " + ", ".join(f"{d} ({v:.0%})" for d, v in zip(x["dominio citato"], x["quota di testate"])))